# TDI 101524 TDRB — Optimized

## What Changed
The original notebook re-scans the same base tables for every metric cell.
Each metric rebuilds identical subqueries for active-customer filtering, joins to reference tables, etc.

**This version materializes shared data once, then each metric is a short query.**

### Architecture
```
Step 1: Cache base tables
  - tdi_barbados_customer (active subset) → used by SegmentData, Centralized, ABAC
  - barbados_registered_office → used by 5.x + ABAC5.1
  - barbados_transaction_fy25 (filtered) → used by SD7/SD8 + 3.x + ABAC3.x
  - barbados_centralized_fy25 → used by 1.x + SD2 + ABAC1.2
Step 2: Cache reference tables (sanctions, country_ref, industry, entity)
Step 3: Run each metric group as short parameterized queries
Step 4: Hardcoded / N/A metrics
Step 5: ABAC metrics
Step 6: Results summary
```

### Metric Groups
| Group | Metrics | Source |
|-------|---------|--------|
| SegmentData | SD1, SD3, 1.6, 1.7, SD4, SD5, 4.1A-C, 6.5A-B | tdi_barbados_customer |
| Address | 5.1-5.9 | barbados_registered_office |
| Transaction | SD7, SD8, 3.1-3.16 | barbados_transaction_fy25 |
| Centralized | 1.1-1.5, SD2, SD6, 1.8, 1.9, 3.17-3.22 | barbados_centralized_fy25 |
| ABAC | ABAC1.1-1.3, ABAC3.1-3.2, ABAC5.1 | cross-table |

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/GAMLConnections

In [ ]:
%run /Workspace/Shared/RiskAssessment/FY_2025/Configs/Settings

---
## Step 1: Cache Base Tables
Each table is loaded ONCE with all columns needed by any metric.

> **Performance:** By caching these views, all metrics share the same pre-filtered dataset.
> The original notebook scans each table multiple times.

In [ ]:
# ============================================================
# Base Table 1: tdi_barbados_customer (active subset)
# Used by: SD1, SD3, 1.6, 1.7, SD4, SD5, 4.1A-C, 6.5B,
#          1.1-1.5 (via subquery), SD6, 1.8, 1.9, 3.17, 3.18,
#          ABAC1.1, ABAC1.3
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW td_bar_cust AS
SELECT *
FROM   ra_adido_2025.101524_tdi_barbados_customer
WHERE  (Length_of_Relationship <= '2025-10-31' AND Customer_Status = 'Active')
   OR  (Length_of_Relationship >= '2024-11-01' AND Customer_Status = 'Inactive')
""")
spark.sql('CACHE TABLE td_bar_cust')

# Also cache the full table for new-customer queries (SD6)
spark.sql("""
CREATE OR REPLACE TEMP VIEW td_bar_cust_full AS
SELECT *
FROM   ra_adido_2025.101524_tdi_barbados_customer
""")
spark.sql('CACHE TABLE td_bar_cust_full')

cnt = spark.sql('SELECT count(1) FROM td_bar_cust').collect()[0][0]
print(f'td_bar_cust (active)     {cnt:>12,} rows')
cnt2 = spark.sql('SELECT count(1) FROM td_bar_cust_full').collect()[0][0]
print(f'td_bar_cust_full         {cnt2:>12,} rows')

In [ ]:
# ============================================================
# Base Table 2: barbados_registered_office
# Used by: 5.1-5.8, ABAC5.1
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW td_bar_office AS
SELECT *
FROM   ra_adido_2025.barbados_registered_office
""")
spark.sql('CACHE TABLE td_bar_office')

cnt = spark.sql('SELECT count(1) FROM td_bar_office').collect()[0][0]
print(f'td_bar_office            {cnt:>12,} rows')

In [ ]:
# ============================================================
# Base Table 3: barbados_transaction_fy25 (filtered)
# Used by: SD7, SD8, 3.1-3.16, ABAC3.1, ABAC3.2
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW td_bar_txn AS
SELECT *
FROM   ra_adido_2025.barbados_transaction_fy25
WHERE  FileName IN ('TD Bank GBP - FY25', 'TD Bank USD - FY25', 'TD Bank EUR - FY25',
                     'TD Bank CAD - FY25', 'TD Bank BBD - FY25')
  AND  (`Incoming-Deposits` IS NOT NULL OR `Outgoing-Payments` IS NOT NULL)
  AND  Jurisdiction != 'Canada'
""")
spark.sql('CACHE TABLE td_bar_txn')

cnt = spark.sql('SELECT count(1) FROM td_bar_txn').collect()[0][0]
print(f'td_bar_txn (filtered)    {cnt:>12,} rows')

In [ ]:
# ============================================================
# Base Table 4: barbados_centralized_fy25
# Used by: 1.1-1.5, SD2, ABAC1.2
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW td_bar_cent AS
SELECT *
FROM   ra_adido_2025.barbados_centralized_fy25
""")
spark.sql('CACHE TABLE td_bar_cent')

cnt = spark.sql('SELECT count(1) FROM td_bar_cent').collect()[0][0]
print(f'td_bar_cent              {cnt:>12,} rows')

In [ ]:
# ============================================================
# Reference Tables: sanctions, country_ref, industry, entity
# ============================================================

spark.sql("""
CREATE OR REPLACE TEMP VIEW ref_sanctions AS
SELECT * FROM ra_adido_2025.sanctions
""")
spark.sql('CACHE TABLE ref_sanctions')

spark.sql("""
CREATE OR REPLACE TEMP VIEW ref_country AS
SELECT * FROM ra_adido_2025.country_ref
""")
spark.sql('CACHE TABLE ref_country')

spark.sql("""
CREATE OR REPLACE TEMP VIEW ref_industry AS
SELECT * FROM ra_adido_2025.industry_ref_list_ca2025
""")
spark.sql('CACHE TABLE ref_industry')

spark.sql("""
CREATE OR REPLACE TEMP VIEW ref_entity AS
SELECT * FROM ra_adido_2025.entity_ref_list_ca2025
""")
spark.sql('CACHE TABLE ref_entity')

# ABAC country risk rating
spark.sql("""
CREATE OR REPLACE TEMP VIEW ref_abac_country AS
SELECT * FROM ra_adido_2025.TD_Country_Risk_Rating_ABAC
""")
spark.sql('CACHE TABLE ref_abac_country')

# Industry reference list for ABAC
spark.sql("""
CREATE OR REPLACE TEMP VIEW ref_industry_abac AS
SELECT * FROM ra_adido_2025.Industry_Reference_List
""")
spark.sql('CACHE TABLE ref_industry_abac')

for v in ['ref_sanctions', 'ref_country', 'ref_industry', 'ref_entity', 'ref_abac_country', 'ref_industry_abac']:
    cnt = spark.sql(f'SELECT count(1) FROM {v}').collect()[0][0]
    print(f'{v:25s} {cnt:>12,} rows')

print('\nAll base + reference tables cached.')

---
## Step 2: SegmentData Metrics (SD1, SD3, 1.6, 1.7, SD4, SD5)
All use `count(distinct Customer_Unique_ID)` from `td_bar_cust` with different WHERE filters.

In [ ]:
# ============================================================
# SD1 — Total active/inactive customer count
# SD3 — Active customers NOT in Barbados
# 1.6 — Customers in HIGH-risk sanctioned countries
# 1.7 — Customers with null/empty/unlisted country code
# ============================================================

results = {}

# SD1: Total customer count (already filtered in td_bar_cust)
sd1 = spark.sql("""
SELECT count(distinct Customer_Unique_ID) FROM td_bar_cust
""").collect()[0][0]
results['SD1'] = sd1
print(f'SD1 (Total Customers):       {sd1:>12,}')

# SD3: Active customers with ISO3 not BRB
sd3 = spark.sql("""
SELECT count(distinct Customer_Unique_ID)
FROM   td_bar_cust
WHERE  Customer_Status IN ('Active')
  AND  ISO3_Country_Code != 'BRB'
""").collect()[0][0]
results['SD3'] = sd3
print(f'SD3 (Non-BRB Active):        {sd3:>12,}')

# 1.6: Customers in HIGH-risk sanctioned countries
cde_1_6 = spark.sql("""
SELECT count(distinct Customer_Unique_ID)
FROM   td_bar_cust
WHERE  ISO3_Country_Code IN (
    SELECT distinct `ISO-ALPHA3`
    FROM   ref_sanctions
    WHERE  RiskRating = 'High'
)
""").collect()[0][0]
results['1.6'] = cde_1_6
print(f'1.6 (High Sanction Country): {cde_1_6:>12,}')

# 1.7: Customers with null/empty/unlisted country code
cde_1_7 = spark.sql("""
SELECT count(distinct Customer_Unique_ID)
FROM   td_bar_cust
WHERE  ISO3_Country_Code IS NULL
   OR  trim(ISO3_Country_Code) = ''
   OR  ISO3_Country_Code NOT IN (
       SELECT distinct `ISO-ALPHA3` FROM ref_sanctions
   )
""").collect()[0][0]
results['1.7'] = cde_1_7
print(f'1.7 (Null/Unlisted Country): {cde_1_7:>12,}')

In [ ]:
# ============================================================
# SD4 — High-risk industry customers
# SD5 — High-risk entity customers
# ============================================================

# SD4: Join customer with industry_ref_list on SIC_Code
sd4 = spark.sql("""
SELECT count(distinct td_bar_cust.Customer_Unique_ID)
FROM   td_bar_cust
JOIN   ref_industry
  ON   td_bar_cust.SIC_Code = ref_industry.Industry_Code
WHERE  ref_industry.Updated_Risk_Rating = 'High'
""").collect()[0][0]
results['SD4'] = sd4
print(f'SD4 (High-Risk Industry):    {sd4:>12,}')

# SD5: Join customer with entity_ref_list on Legal_Entity_Classification
sd5 = spark.sql("""
SELECT count(distinct td_bar_cust.Customer_Unique_ID)
FROM   td_bar_cust
JOIN   ref_entity
  ON   td_bar_cust.Legal_Entity_Classification = ref_entity.Entity_Type_Name
WHERE  ref_entity.Updated_Risk_Rating = 'High'
""").collect()[0][0]
results['SD5'] = sd5
print(f'SD5 (High-Risk Entity):      {sd5:>12,}')

In [ ]:
# ============================================================
# 4.1A — New customers (within assessment period)
# 4.1B — New customers using VIRTUAL delivery channel
# 4.1C — New customers with null delivery channel
# ============================================================

# 4.1A: New customers within date range
cde_4_1a = spark.sql("""
SELECT count(distinct Customer_Unique_ID)
FROM   td_bar_cust
WHERE  Length_of_Relationship >= '2024-11-01'
  AND  Length_of_Relationship <= '2025-10-31'
""").collect()[0][0]
results['4.1A'] = cde_4_1a
print(f'4.1A (New Customers):        {cde_4_1a:>12,}')

# 4.1B: New customers + VIRTUAL delivery channel
cde_4_1b = spark.sql("""
SELECT count(distinct Customer_Unique_ID)
FROM   td_bar_cust
WHERE  Length_of_Relationship >= '2024-11-01'
  AND  Length_of_Relationship <= '2025-10-31'
  AND  DeliveryChannel = 'VIRTUAL'
""").collect()[0][0]
results['4.1B'] = cde_4_1b
print(f'4.1B (New + Virtual):        {cde_4_1b:>12,}')

# 4.1C: New customers + null delivery channel
cde_4_1c = spark.sql("""
SELECT count(distinct Customer_Unique_ID)
FROM   td_bar_cust
WHERE  Length_of_Relationship >= '2024-11-01'
  AND  Length_of_Relationship <= '2025-10-31'
  AND  DeliveryChannel IS NULL
""").collect()[0][0]
results['4.1C'] = cde_4_1c
print(f'4.1C (New + Null Channel):   {cde_4_1c:>12,}')

In [ ]:
# ============================================================
# 6.5A — Prior year CDE result
# 6.5B — Active customer count
# ============================================================

# 6.5A: Lookup from prior year results
cde_6_5a = spark.sql("""
SELECT values AS `6_5_A`
FROM   ra_adhoc_data.101524_cde_results_fy2024
WHERE  CDE = '1'
""").collect()[0][0]
results['6.5A'] = cde_6_5a
print(f'6.5A (Prior Year):           {cde_6_5a}')

# 6.5B: Active customer count
cde_6_5b = spark.sql("""
SELECT count(distinct Customer_Unique_ID)
FROM   td_bar_cust
WHERE  Customer_Status = 'Active'
""").collect()[0][0]
results['6.5B'] = cde_6_5b
print(f'6.5B (Active Customers):     {cde_6_5b:>12,}')

---
## Step 3: Address Metrics (5.1–5.9)
All use `count(distinct address)` from `td_bar_office`, joined with `country_ref` (5.2-5.4) and `sanctions` (5.5-5.8).

In [ ]:
# ============================================================
# 5.1 — Total distinct registered office addresses
# 5.2-5.4 — By country_ref risk rating (High, Medium, Low)
# 5.5-5.8 — By sanctions risk rating (Very High, High, Medium, Low)
# 5.9 — Not Applicable (hardcoded)
# ============================================================

# 5.1: Total distinct addresses
cde_5_1 = spark.sql("""
SELECT count(distinct address)
FROM   td_bar_office
""").collect()[0][0]
results['5.1'] = cde_5_1
print(f'5.1 (Total Addresses):       {cde_5_1:>12,}')

# 5.2-5.4: By country_ref risk rating
country_risk_metrics = [
    ('5.2', 'High'),
    ('5.3', 'Medium'),
    ('5.4', 'Low'),
]
for metric_id, risk_level in country_risk_metrics:
    cnt = spark.sql(f"""
    SELECT count(distinct o.address)
    FROM   td_bar_office o
    JOIN   ref_country c ON o.ISO3_Country_Code = c.ISO3_Country_Code
    WHERE  c.RiskRating = '{risk_level}'
    """).collect()[0][0]
    results[metric_id] = cnt
    print(f'{metric_id} (Country {risk_level:6s}):       {cnt:>12,}')

# 5.5-5.8: By sanctions risk rating
sanctions_risk_metrics = [
    ('5.5', 'Very High'),
    ('5.6', 'High'),
    ('5.7', 'Medium'),
    ('5.8', 'Low'),
]
for metric_id, risk_level in sanctions_risk_metrics:
    cnt = spark.sql(f"""
    SELECT count(distinct o.address)
    FROM   td_bar_office o
    JOIN   ref_sanctions s ON o.ISO3_Country_Code = s.`ISO-ALPHA3`
    WHERE  s.RiskRating = '{risk_level}'
    """).collect()[0][0]
    results[metric_id] = cnt
    print(f'{metric_id} (Sanctions {risk_level:9s}):  {cnt:>12,}')

# 5.9: Not Applicable
results['5.9'] = 'Not Applicable'
print(f'5.9:                         Not Applicable')

---
## Step 4: Transaction Metrics (SD7, SD8, 3.1–3.16)
All use the cached `td_bar_txn` view (already filtered for relevant FileNames and non-null amounts).

In [ ]:
# ============================================================
# SD7 — Transaction count
# SD8 — Net transaction amount
# ============================================================

# SD7: Count of transactions
sd7 = spark.sql("""
SELECT count(*) FROM td_bar_txn
""").collect()[0][0]
results['SD7'] = sd7
print(f'SD7 (Transaction Count):     {sd7:>12,}')

# SD8: Net amount (Incoming - Outgoing)
sd8 = spark.sql("""
SELECT round(sum(`Incoming-Deposits`) - sum(`Outgoing-Payments`), 2)
FROM   td_bar_txn
""").collect()[0][0]
results['SD8'] = sd8
print(f'SD8 (Net Amount):            {sd8:>12}')

In [ ]:
# ============================================================
# 3.1-3.4 — Transaction count by country_ref risk rating
# 3.1 = Unknown, 3.2 = High, 3.3 = Medium, 3.4 = Low
# ============================================================

# 3.1: Unknown (no match in country_ref)
cde_3_1 = spark.sql("""
SELECT count(*)
FROM   td_bar_txn t
LEFT JOIN ref_country c ON t.ISO3_Country_Code = c.ISO3_Country_Code
WHERE  c.ISO3_Country_Code IS NULL
""").collect()[0][0]
results['3.1'] = cde_3_1
print(f'3.1 (Country Unknown):       {cde_3_1:>12,}')

# 3.2-3.4: By country risk rating
txn_country_metrics = [
    ('3.2', 'High'),
    ('3.3', 'Medium'),
    ('3.4', 'Low'),
]
for metric_id, risk_level in txn_country_metrics:
    cnt = spark.sql(f"""
    SELECT count(*)
    FROM   td_bar_txn t
    JOIN   ref_country c ON t.ISO3_Country_Code = c.ISO3_Country_Code
    WHERE  c.RiskRating = '{risk_level}'
    """).collect()[0][0]
    results[metric_id] = cnt
    print(f'{metric_id} (Country {risk_level:6s}):       {cnt:>12,}')

In [ ]:
# ============================================================
# 3.5-3.8 — Transaction count by sanctions risk rating
# 3.5 = Very High, 3.6 = High, 3.7 = Medium, 3.8 = Low
# ============================================================

txn_sanctions_metrics = [
    ('3.5', 'Very High'),
    ('3.6', 'High'),
    ('3.7', 'Medium'),
    ('3.8', 'Low'),
]
for metric_id, risk_level in txn_sanctions_metrics:
    cnt = spark.sql(f"""
    SELECT count(*)
    FROM   td_bar_txn t
    JOIN   ref_sanctions s ON t.ISO3_Country_Code = s.`ISO-ALPHA3`
    WHERE  s.RiskRating = '{risk_level}'
    """).collect()[0][0]
    results[metric_id] = cnt
    print(f'{metric_id} (Sanctions {risk_level:9s}):  {cnt:>12,}')

In [ ]:
# ============================================================
# 3.9-3.12 — Net amount by country_ref risk rating
# 3.9 = Unknown, 3.10 = High, 3.11 = Medium, 3.12 = Low
# ============================================================

# 3.9: Unknown (no match in country_ref)
cde_3_9 = spark.sql("""
SELECT round(sum(`Incoming-Deposits`) - sum(`Outgoing-Payments`), 2)
FROM   td_bar_txn t
LEFT JOIN ref_country c ON t.ISO3_Country_Code = c.ISO3_Country_Code
WHERE  c.ISO3_Country_Code IS NULL
""").collect()[0][0]
results['3.9'] = cde_3_9
print(f'3.9 (Amount Unknown):        {cde_3_9}')

# 3.10-3.12: By country risk rating
amt_country_metrics = [
    ('3.10', 'High'),
    ('3.11', 'Medium'),
    ('3.12', 'Low'),
]
for metric_id, risk_level in amt_country_metrics:
    amt = spark.sql(f"""
    SELECT round(sum(`Incoming-Deposits`) - sum(`Outgoing-Payments`), 2)
    FROM   td_bar_txn t
    JOIN   ref_country c ON t.ISO3_Country_Code = c.ISO3_Country_Code
    WHERE  c.RiskRating = '{risk_level}'
    """).collect()[0][0]
    results[metric_id] = amt
    print(f'{metric_id} (Amount {risk_level:6s}):       {amt}')

In [ ]:
# ============================================================
# 3.13-3.16 — Net amount by sanctions risk rating
# 3.13 = Very High, 3.14 = High, 3.15 = Medium, 3.16 = Low
# ============================================================

amt_sanctions_metrics = [
    ('3.13', 'Very High'),
    ('3.14', 'High'),
    ('3.15', 'Medium'),
    ('3.16', 'Low'),
]
for metric_id, risk_level in amt_sanctions_metrics:
    amt = spark.sql(f"""
    SELECT round(sum(`Incoming-Deposits`) - sum(`Outgoing-Payments`), 2)
    FROM   td_bar_txn t
    JOIN   ref_sanctions s ON t.ISO3_Country_Code = s.`ISO-ALPHA3`
    WHERE  s.RiskRating = '{risk_level}'
    """).collect()[0][0]
    results[metric_id] = amt
    print(f'{metric_id} (Amount {risk_level:9s}):    {amt}')

---
## Step 5: Centralized Metrics (1.1–1.5, SD2, SD6, 1.8, 1.9, 3.17–3.22)
These use the cached `td_bar_cent` view and join with the active customer subquery.

In [ ]:
# ============================================================
# 1.1-1.5 — Customer risk rating from centralized
# All use: count(distinct Customer_Unique_ID) from td_bar_cent
# WHERE Customer_Unique_ID IN (active customers from td_bar_cust)
# 1.1 = null risk, 1.2 = Tier1/2, 1.3 = High, 1.4 = Medium, 1.5 = Low
# ============================================================

# Active customer subquery (reused across 1.1-1.5)
active_subquery = """
    SELECT DISTINCT Customer_Unique_ID
    FROM   td_bar_cust
    WHERE  Customer_Status = 'Active'
"""

cent_risk_metrics = [
    ('1.1', 'TDRBRISKRATING IS NULL'),
    ('1.2', "TDRBRISKRATING IN ('Tier 1', 'Tier 2')"),
    ('1.3', "TDRBRISKRATING = 'High'"),
    ('1.4', "TDRBRISKRATING = 'Medium'"),
    ('1.5', "TDRBRISKRATING = 'Low'"),
]

for metric_id, risk_filter in cent_risk_metrics:
    cnt = spark.sql(f"""
    SELECT count(distinct Customer_Unique_ID)
    FROM   td_bar_cent
    WHERE  Customer_Unique_ID IN ({active_subquery})
      AND  {risk_filter}
    """).collect()[0][0]
    results[metric_id] = cnt
    print(f'{metric_id} ({risk_filter:40s}): {cnt:>12,}')

In [ ]:
# ============================================================
# SD2 — PEP / Hits based on alerts
# From centralized WHERE PEP or alert-hit flags = 'YES'
# ============================================================

sd2 = spark.sql(f"""
SELECT count(distinct Customer_Unique_ID)
FROM   td_bar_cent
WHERE  Customer_Unique_ID IN ({active_subquery})
  AND  (POLITICALLYEXPOSEDPERSONS = 'YES' OR HITsbasedonalerts = 'YES')
""").collect()[0][0]
results['SD2'] = sd2
print(f'SD2 (PEP/Alerts):            {sd2:>12,}')

In [ ]:
# ============================================================
# SD6 — New customers (date range filter on customer table)
# count(distinct Customer_Unique_ID) from centralized
# WHERE Customer_Unique_ID in new-customers subquery
# ============================================================

sd6 = spark.sql("""
SELECT count(distinct Customer_Unique_ID)
FROM   td_bar_cust_full
WHERE  Length_of_Relationship >= '2024-11-01'
  AND  Length_of_Relationship <= '2025-10-31'
""").collect()[0][0]
results['SD6'] = sd6
print(f'SD6 (New Customers):         {sd6:>12,}')

In [ ]:
# ============================================================
# 1.8 — True sanction match (levenshtein)
# JOIN tdi_barbados_customer with true_sanction_match_2025
# WHERE levenshtein(Customer_Name_Beneficial_Owner, CustomerName) <= 5
# AND CustomerType = 'Non-Personal'
# ============================================================

cde_1_8 = spark.sql("""
SELECT count(*)
FROM   td_bar_cust c
JOIN   ra_adido_2025.true_sanction_match_2025 s
  ON   levenshtein(c.Customer_Name_Beneficial_Owner, s.CustomerName) <= 5
WHERE  s.CustomerType = 'Non-Personal'
""").collect()[0][0]
results['1.8'] = cde_1_8
print(f'1.8 (True Sanction Match):   {cde_1_8:>12,}')

In [ ]:
# ============================================================
# 1.9 — Sanctioned blocked property (levenshtein)
# JOIN tdi_barbados_customer with customer_sanctioned_blocked_property_2025
# WHERE levenshtein(Customer_Name_Beneficial_Owner, ...) <= 5
# ============================================================

cde_1_9 = spark.sql("""
SELECT count(*)
FROM   td_bar_cust c
JOIN   ra_adido_2025.customer_sanctioned_blocked_property_2025 s
  ON   levenshtein(c.Customer_Name_Beneficial_Owner, s.CustomerName) <= 5
""").collect()[0][0]
results['1.9'] = cde_1_9
print(f'1.9 (Blocked Property):      {cde_1_9:>12,}')

In [ ]:
# ============================================================
# 3.17 — UTR (Unusual Transaction Report) via levenshtein
# JOIN tdi_barbados_customer with barbados_utr
# WHERE levenshtein(Customer_Name_Beneficial_Owner, details) <= 5
# ============================================================

cde_3_17 = spark.sql("""
SELECT count(*)
FROM   td_bar_cust c
JOIN   ra_adido_2025.barbados_utr u
  ON   levenshtein(c.Customer_Name_Beneficial_Owner, u.details) <= 5
""").collect()[0][0]
results['3.17'] = cde_3_17
print(f'3.17 (UTR):                  {cde_3_17:>12,}')

In [ ]:
# ============================================================
# 3.18 — STR (Suspicious Transaction Report) via levenshtein
# JOIN tdi_barbados_customer with barbados_str
# WHERE levenshtein(Customer_Name_Beneficial_Owner, ...) <= 5
# ============================================================

cde_3_18 = spark.sql("""
SELECT count(*)
FROM   td_bar_cust c
JOIN   ra_adido_2025.barbados_str s
  ON   levenshtein(c.Customer_Name_Beneficial_Owner, s.details) <= 5
""").collect()[0][0]
results['3.18'] = cde_3_18
print(f'3.18 (STR):                  {cde_3_18:>12,}')

In [ ]:
# ============================================================
# Hardcoded / N/A Metrics
# 3.19 = 'Not Applicable'
# 3.20-3.22 = '0'
# ============================================================

results['3.19'] = 'Not Applicable'
results['3.20'] = 0
results['3.21'] = 0
results['3.22'] = 0

print(f'3.19:                        Not Applicable')
print(f'3.20:                        0')
print(f'3.21:                        0')
print(f'3.22:                        0')

---
## Step 6: ABAC Metrics
Anti-Bribery / Anti-Corruption metrics using cross-table joins.

In [ ]:
# ============================================================
# ABAC1.1 — High ABAC country risk ratio
# sum(case when FinalABACRating = 'High' then 1 else 0 end)
#   / count(distinct Customer_Unique_ID) * 100
# FROM tdi_barbados_customer JOIN country_ref JOIN TD_Country_Risk_Rating_ABAC
# ============================================================

abac_1_1 = spark.sql("""
SELECT round(
    sum(CASE WHEN abac.FinalABACRating = 'High' THEN 1 ELSE 0 END)
    / count(distinct c.Customer_Unique_ID) * 100,
    2
)
FROM   td_bar_cust c
JOIN   ref_country cr ON c.ISO3_Country_Code = cr.ISO3_Country_Code
JOIN   ref_abac_country abac ON cr.ISO3_Country_Code = abac.ISO3_Country_Code
""").collect()[0][0]
results['ABAC1.1'] = abac_1_1
print(f'ABAC1.1 (Country Risk %):    {abac_1_1}')

In [ ]:
# ============================================================
# ABAC1.2 — PEP count from centralized
# count(distinct Customer_Unique_ID) from barbados_centralized
# WHERE POLITICALLYEXPOSEDPERSONS = 'YES'
# ============================================================

abac_1_2 = spark.sql(f"""
SELECT count(distinct Customer_Unique_ID)
FROM   td_bar_cent
WHERE  Customer_Unique_ID IN ({active_subquery})
  AND  POLITICALLYEXPOSEDPERSONS = 'YES'
""").collect()[0][0]
results['ABAC1.2'] = abac_1_2
print(f'ABAC1.2 (PEP Count):         {abac_1_2:>12,}')

In [ ]:
# ============================================================
# ABAC1.3 — High ABAC industry risk ratio
# sum(case when USABACTeam = 'High' then 1 else 0 end)
#   / count(distinct Customer_Unique_ID) * 100
# FROM tdi_barbados_customer JOIN Industry_Reference_List ON Standardized_SIC_Code
# ============================================================

abac_1_3 = spark.sql("""
SELECT round(
    sum(CASE WHEN i.USABACTeam = 'High' THEN 1 ELSE 0 END)
    / count(distinct c.Customer_Unique_ID) * 100,
    2
)
FROM   td_bar_cust c
JOIN   ref_industry_abac i ON c.Standardized_SIC_Code = i.Standardized_SIC_Code
""").collect()[0][0]
results['ABAC1.3'] = abac_1_3
print(f'ABAC1.3 (Industry Risk %):   {abac_1_3}')

In [ ]:
# ============================================================
# ABAC5.1 — High ABAC risk registered office address count
# count(distinct address) from barbados_registered_office
# WHERE ABAC rating = High
# ============================================================

abac_5_1 = spark.sql("""
SELECT count(distinct o.address)
FROM   td_bar_office o
JOIN   ref_abac_country abac ON o.ISO3_Country_Code = abac.ISO3_Country_Code
WHERE  abac.FinalABACRating = 'High'
""").collect()[0][0]
results['ABAC5.1'] = abac_5_1
print(f'ABAC5.1 (ABAC High Addr):    {abac_5_1:>12,}')

In [ ]:
# ============================================================
# ABAC3.1 — Transaction High ABAC risk ratio (count-based)
# sum(case when FinalABACRating = 'High' then 1 else 0 end)
#   / count(*) * 100
# FROM barbados_transaction JOIN TD_Country_Risk_Rating_ABAC
# ============================================================

abac_3_1 = spark.sql("""
SELECT round(
    sum(CASE WHEN abac.FinalABACRating = 'High' THEN 1 ELSE 0 END)
    / count(*) * 100,
    2
)
FROM   td_bar_txn t
JOIN   ref_abac_country abac ON t.ISO3_Country_Code = abac.ISO3_Country_Code
""").collect()[0][0]
results['ABAC3.1'] = abac_3_1
print(f'ABAC3.1 (Txn Risk %):        {abac_3_1}')

In [ ]:
# ============================================================
# ABAC3.2 — Transaction High ABAC risk ratio (dollar-weighted)
# Dollar-weighted ratio from same
# ============================================================

abac_3_2 = spark.sql("""
SELECT round(
    sum(CASE WHEN abac.FinalABACRating = 'High'
        THEN coalesce(`Incoming-Deposits`, 0) + coalesce(`Outgoing-Payments`, 0)
        ELSE 0 END)
    / sum(coalesce(`Incoming-Deposits`, 0) + coalesce(`Outgoing-Payments`, 0)) * 100,
    2
)
FROM   td_bar_txn t
JOIN   ref_abac_country abac ON t.ISO3_Country_Code = abac.ISO3_Country_Code
""").collect()[0][0]
results['ABAC3.2'] = abac_3_2
print(f'ABAC3.2 (Txn Dollar %):      {abac_3_2}')

---
## Results Summary
All metrics collected above.

In [ ]:
# ============================================================
# Final Results Summary
# ============================================================
print('=' * 60)
print('TDI 101524 TDRB — Results Summary')
print('=' * 60)
print()

# Define display order
metric_groups = [
    ('SegmentData', ['SD1', 'SD3', '1.6', '1.7', 'SD4', 'SD5',
                     '4.1A', '4.1B', '4.1C', '6.5A', '6.5B']),
    ('Address',     ['5.1', '5.2', '5.3', '5.4', '5.5', '5.6',
                     '5.7', '5.8', '5.9']),
    ('Transaction', ['SD7', 'SD8', '3.1', '3.2', '3.3', '3.4',
                     '3.5', '3.6', '3.7', '3.8',
                     '3.9', '3.10', '3.11', '3.12',
                     '3.13', '3.14', '3.15', '3.16']),
    ('Centralized', ['1.1', '1.2', '1.3', '1.4', '1.5',
                     'SD2', 'SD6', '1.8', '1.9',
                     '3.17', '3.18', '3.19', '3.20', '3.21', '3.22']),
    ('ABAC',        ['ABAC1.1', 'ABAC1.2', 'ABAC1.3',
                     'ABAC5.1', 'ABAC3.1', 'ABAC3.2']),
]

for group_name, metric_ids in metric_groups:
    print(f'\n--- {group_name} ---')
    for mid in metric_ids:
        val = results.get(mid, 'N/A')
        if isinstance(val, (int, float)):
            print(f'  {mid:12s} {val:>15,}')
        else:
            print(f'  {mid:12s} {str(val):>15s}')

print(f'\nTotal metrics: {len(results)}')
print('=' * 60)

---
## Cleanup (Optional)
Uncache temporary views to free memory.

In [ ]:
# ============================================================
# Uncache all temp views
# ============================================================
cached_views = [
    'td_bar_cust', 'td_bar_cust_full',
    'td_bar_office', 'td_bar_txn', 'td_bar_cent',
    'ref_sanctions', 'ref_country', 'ref_industry', 'ref_entity',
    'ref_abac_country', 'ref_industry_abac',
]
for v in cached_views:
    try:
        spark.sql(f'UNCACHE TABLE {v}')
        print(f'Uncached {v}')
    except:
        pass
print('\nCleanup complete.')